### Track C - Integrate Pybullet simulation with Pytorch model

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset


import pybullet as p
import pybullet_data

import matplotlib.pyplot as plt
import math
import time
from pathlib import Path
from PIL import Image
import numpy as np
import h5py


pybullet build time: Nov 28 2023 23:45:17


### C1. Simplify the task drastically

1. Reach task

In [2]:
SEED=5
torch.manual_seed(SEED)
np.random.seed(SEED)
# p.disconnect()

In [3]:
PANDA_ARM_JOINTS = [0, 1, 2, 3, 4, 5, 6]
PANDA_FINGER_JOINTS = [9, 10]


class MiniArmEnv():
    '''
    TODO: A function that adjusts the size of delta_step depending on the distance between end-effector and sphere.
            -- Large steps when it's far. Small steps when it's close.
    '''
    def __init__(self, max_steps: int = 300, render: bool = True):
        self.max_steps = max_steps
        self.physics_engine_update = 240 # 240 Hz.
        self.success_threshold = 0.25
        self._step_count = 0
        self.delta_step = 0.05 # update 5cm towards the direction of the sphere per step.
        
        mode = p.GUI if render else p.DIRECT
        self.client = p.connect(mode)

        # load scene
        self._load_scene()

        # get end effector index.
        self.end_effector = self._get_end_effector(robotId=self.robotId)

    def reset(self):
        home_positions = [0, -0.785, 0, -2.356, 0, 1.571, 0.785]
        finger_home = [0.04, 0.04]  # open gripper

        for j, angle in zip(PANDA_ARM_JOINTS, home_positions):
            p.resetJointState(self.robotId, j, angle, targetVelocity=0)

        for j, angle in zip(PANDA_FINGER_JOINTS, finger_home):
            p.resetJointState(self.robotId, j, angle, targetVelocity=0)

        # reset sphere position
        p.resetBasePositionAndOrientation(bodyUniqueId=self.sphereId,
                                          posObj=[np.random.uniform(-0.4, 0.4),
                                                  np.random.uniform(-0.4, 0.4),
                                                  np.random.uniform(0.4, 0.4)],
                                          ornObj=[0.0, 0.0, 0.0, 1.0])
        
        self.sphereId_position = np.array(p.getBasePositionAndOrientation(self.sphereId)[0])
        
        p.stepSimulation()
        return self._get_obs()

    def step(self, action):
        
        self._apply_action(action)

        p.setTimeStep(1 / self.physics_engine_update)

        for _ in range(18):
            # TODO: add a break. If the EE reached the target position it stops. Use self._check_ee_close_sphere() every N steps.
            p.stepSimulation()

        self._step_count += 1        
                  
        obs = self._get_obs()
        reward = None
        done = self._is_done() or self._step_count >= self.max_steps
        info = {"step": self._step_count, "success": self._check_position_success()}

        return obs, reward, done, info

    def close(self):
        p.disconnect(self.client)


    # ── scene loading ────────────────────────────────────────────────────────
    def _load_scene(self):
        "Load plane, frank arm and sphere"
        
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0, 0, -9.81)

        p.loadURDF("plane.urdf")
        # upload the franka arm.
        self.robotId = p.loadURDF("franka_panda/panda.urdf",
                                basePosition=[0,0,0],
                                useFixedBase=True)

        # create a sphere
        radius = 0.1
        visual_shape = p.createVisualShape(shapeType=p.GEOM_SPHERE,
                                        radius=radius,
                                        rgbaColor=[0, 1, 0, 1])

        collision_shape = p.createCollisionShape(shapeType=p.GEOM_SPHERE,
                                                radius=radius)
                                                
        self.sphereId = p.createMultiBody(baseMass=0,
                                    baseCollisionShapeIndex=collision_shape,
                                    baseVisualShapeIndex=visual_shape,
                                    basePosition=[1,1,1])

    # ── action application ───────────────────────────────────────────────────

    def _apply_action(self, action):
        ee_pos, ee_orn = self._get_ee_pose() # reach task: the end effector must reach the position of the sphere.
        delta_pos = action * self.delta_step # delta (x,y,z) in meters (MKS system).
        target_pos = ee_pos + delta_pos
        
        # inverse kinematics
        joint_angles = p.calculateInverseKinematics(
            bodyUniqueId=self.robotId,
            endEffectorLinkIndex=self.end_effector,
            targetPosition=target_pos, # TODO: Add the ee_orn???
            maxNumIterations=400,
            residualThreshold=1e-7
        )

        # move the arm
        p.setJointMotorControlArray(
            bodyIndex=self.robotId,
            jointIndices=PANDA_ARM_JOINTS + PANDA_FINGER_JOINTS,
            controlMode=p.POSITION_CONTROL,
            targetPositions = joint_angles,
            forces = [500]*len(PANDA_ARM_JOINTS + PANDA_FINGER_JOINTS)
        )


    # ── observations ─────────────────────────────────────────────────────────

    def _get_obs(self):
        rgb_arr, depth_arr = self._render_camera()
        proprio = self._get_proprio()
        ee_pos, ee_orn = self._get_ee_pose() # position of robotic arm end effector in the URDF link frame (the one placed at a meaningful reference point by the URDF designer)
        sphere_pos = self.sphereId_position
        
        return {
            "rgb": rgb_arr,
            "depth": depth_arr,
            "proprio": proprio,
            "ee_pos": ee_pos,
            "ee_orn": ee_orn,
            "sphere_pos": sphere_pos
        }

    def _render_camera(self):
        view_mat = p.computeViewMatrix(
            cameraEyePosition    = [1,0,2], 
            cameraTargetPosition = [0,0,1],
            cameraUpVector       = [0,1,0.5],
            physicsClientId     = self.client
            )

        proj_mat = p.computeProjectionMatrixFOV(
            fov=60,
            aspect=1.0,
            nearVal=0.1,
            farVal=10
            )
                                        
        _, _, rgb, depth, _ = p.getCameraImage(
            width=1200,
            height=1200,
            viewMatrix = view_mat,
            projectionMatrix=proj_mat,
            physicsClientId = self.client
        )
        rgb_arr   = np.array(rgb,  dtype=np.uint8)[:, :, :3]
        depth_arr = np.array(depth, dtype=np.float32)[..., None]

        return rgb_arr, depth_arr

    def _get_proprio(self) -> np.ndarray:
        '''
        * joint_states has length = PANDA_ARM_JOINTS
        * s[0] position value of the joint "s"
        * s[1] velocity value of the joint "s"
        '''
        joint_states = p.getJointStates(
            self.robotId, PANDA_ARM_JOINTS,
            physicsClientId=self.client
        )
        q   = np.array([s[0] for s in joint_states], dtype=np.float32) # joint angle (rad): how much a joint is rotated. Internal configuration.
        dq  = np.array([s[1] for s in joint_states], dtype=np.float32) # velocity
        ee_pos, ee_orn = self._get_ee_pose() # ee_pos: position (x,y,z) of the end effector.
        return np.concatenate([q, dq, ee_pos, ee_orn]).astype(np.float32)
    
    def _get_ee_pose(self):
        state = p.getLinkState(
            self.robotId, self.end_effector,
            physicsClientId=self.client
        )
        return np.array(state[4]), state[5]   # pos, orn (quaternion) in URDF frame of reference.

    # ── reward & termination ─────────────────────────────────────────────────
    def _check_position_success(self):
        return np.sqrt(np.dot(self._get_obs()["ee_pos"] - self.sphereId_position, self._get_obs()["ee_pos"] - self.sphereId_position)) < self.success_threshold

    def _is_done(self):
        if self._check_position_success():
            print(f"End-effector is close (< {self.success_threshold} m) to sphere position")
        return self._check_position_success()

    # ── &&&&&& ─────────────────────────────────────────────────
    @staticmethod
    def _get_end_effector(robotId: int):
        end_effector_index = None

        for i in range(p.getNumJoints(robotId)):
            link_name = p.getJointInfo(robotId, i)[12].decode("utf-8")

            if link_name == "panda_hand":
                end_effector_index = i
                break
        return end_effector_index


In [58]:
arm_env.close()
arm_env = MiniArmEnv()
arm_env.reset()
dt_pos = np.array([0.0,0.0,0.01]) # delta perturbation in the position.
obs, reward, done, info = arm_env.step(action=dt_pos)
print(arm_env._step_count)
print(obs)
print(done)

error: Not connected to physics server.

### C2. Oracle script & C3. Collect a dataset

In [4]:
class Oracle():
    def reset(self, sphere_pos: np.ndarray):
        '''Adds as attributes to the Oracle object the position of the sphere.
           Store here the target positions. Those that you do not want to change, but rather reached.'''
        self.sphere_pos = sphere_pos.copy()
    
    def act(self, obs: dict):
        current_ee_pos = obs["ee_pos"]
        target = self.sphere_pos
        action = self._delta_to(current_ee_pos, target)
        return action

    def _delta_to(self, current: np.ndarray, target: np.ndarray):
        "Compute the delta position to then move"
        diff = target - current
        dist = np.linalg.norm(diff)
        # normalise the action. delta only provides the direction of adjustment.
        delta = diff / (np.linalg.norm(diff) + 1e-8) # + 1e-8 controls an error arising from np.linalg.norm(diff) = 0.
        return delta

def collect_demos(n_demos: int, out_path: str, domain: str = "source"):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    env = MiniArmEnv(max_steps=50)
    oracle = Oracle()
    
    rgb_list, depth_list, proprio_list, ee_pos_list, ee_orn_list, actions_list = [], [], [], [], [], []
    collected = 0

    while collected < n_demos:
        obs = env.reset()
        oracle.reset(obs["sphere_pos"])

        rgb, depth, proprio, ee_pos, ee_orn, actions = [], [], [], [], [], []


        done = False
        while not done:
            action = oracle.act(obs) # generates the action as a delta between current end-effector position and sphere position.
            
            rgb.append(obs["rgb"])
            depth.append(obs["depth"])
            proprio.append(obs["proprio"])
            ee_pos.append(obs["ee_pos"])
            ee_orn.append(obs["ee_orn"])
            actions.append(action)

            obs, _, done, info = env.step(action)
            print(done)
        if info["success"]:
            
            rgb_list.append(np.stack(rgb))
            depth_list.append(np.stack(depth))
            proprio_list.append(np.stack(proprio))
            ee_pos_list.append(np.stack(ee_pos))
            ee_orn_list.append(np.stack(ee_orn))
            actions_list.append(np.stack(actions))

            collected += 1
        
        print(f"Current demo: {collected}\t\tSteps: {info['step']}")
    
    env.close()
    
    _save_hdf5(out_path, rgb_list, depth_list, proprio_list, ee_pos_list, ee_orn_list, actions_list)


def _save_hdf5(out_path: str, rgb: list, depth: list, proprio: list, ee_pos: list, ee_orn: list, actions: list):
    with h5py.File(out_path, "w") as f:
        for i, (r,d,p,ep,eo,a) in enumerate(zip(rgb, depth, proprio, ee_pos, ee_orn, actions)):
            grp = f.create_group(f"demo_{i:04d}")
            grp.create_dataset("rgb", data=r, compression="gzip")
            grp.create_dataset("depth", data=d, compression="gzip")
            grp.create_dataset("proprio", data=p, compression="gzip")
            grp.create_dataset("ee_pos", data=ep, compression="gzip")
            grp.create_dataset("ee_orn", data=eo, compression="gzip")
            grp.create_dataset("actions", data=a, compression="gzip")

In [82]:
# p.disconnect()
collect_demos(n_demos=3, out_path=Path.cwd() / "data" / "demos.hdf5")

startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Intel
GL_RENDERER=Mesa Intel(R) Graphics (MTL)
GL_VERSION=4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
GL_SHADING_LANGUAGE_VERSION=4.60
pthread_getconcurrency()=0
Version = 4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
Vendor = Intel
Renderer = Mesa Intel(R) Graphics (MTL)
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
False
False
False
False
False
False
False
False
False
False
False
False
False
End-effector is close (< 0.25 m) to sphere position
True
Current demo: 1		Steps: 14
False
False
False
Fa

TypeError: _save_hdf5() missing 1 required positional argument: 'actions'

### C4. Build your dataset class

* Split into train and validation by splitting between demonstrations (i.e., if samples of demo_0000 are in train, they should not be in validation dataset).
    * Otherwise, you might run into leakage problem. That is, concluding that generalization is happening (good performance in validation dataset) without acknowledging that the model might just being able to predict what had already been seen.
    * Can the model perform well on demonstrations it has never seen before?


In [ ]:
class ReachDataset(Dataset):
    """
    Loads all demonstrations from an HDF5 file into memory and returns
    individual timesteps (not full episodes).
    """

    def __init__(self, hdf5_path: str):
        self.samples: list[dict] = [] # self.samples contains lists with dictionaries inside.

        with h5py.File(hdf5_path, "r") as f:
            demo_keys = sorted(f.keys()) # n elements, with n = number of demos.
            for episode_idx, key in enumerate(demo_keys[:]):
                grp     = f[key]
                n_steps = grp["proprio"].shape[0]
                rgb     = grp["rgb"][:]         
                depth   = grp["depth"][:]       
                proprio = grp["proprio"][:]     
                ee_pos  = grp["ee_pos"][:]
                # actions = grp["actions"][:]

                # for-loop flattens episodes into individual timesteps samples.
                for t in range(n_steps):
                    self.samples.append({
                        "rgb":          rgb[t],
                        "depth":        depth[t],
                        "proprio":      proprio[t],
                        "ee_pos":       ee_pos[t],
                        "episode_idx":  episode_idx # allows keeping track from which episode the samples come.
                    })


    '''
    These are internal methods used when applying to object = DemoDataset():
    len(object) --> __len__()
    object[idx] --> __getitem__()
    '''
    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        return self.samples[idx]


def collate_fn(batch):
    '''
    Stack multiple samples (one sample is one step of one episode).
    '''
    return {
        "rgb": torch.from_numpy(
            np.stack([sample["rgb"] for sample in batch])
        ),

        "depth": torch.from_numpy(
            np.stack([sample["depth"] for sample in batch])
        ),

        "proprio": torch.from_numpy(
            np.stack([sample["proprio"] for sample in batch])
        ),

        "ee_pos": torch.from_numpy(
            np.stack([sample["ee_pos"] for sample in batch])
        ),

        "episode_idx": torch.tensor(
            [sample["episode_idx"] for sample in batch],
            dtype=torch.long),
    }

# Is there intersection between the episodes in train and val set?
def _check_intersection_episodes(train_set, val_set):
    train_episode_ids = []
    val_episode_ids = []

    for i in range(len(train_set)):
        train_episode_ids.append(train_set[i]["episode_idx"])

    for i in range(len(val_set)):
            val_episode_ids.append(val_set[i]["episode_idx"])

    train_episode_ids = set(train_episode_ids)
    val_episode_ids = set(val_episode_ids)
    print(f"train episodes: {train_episode_ids} \t\t val episode: {val_episode_ids}")
    return len(train_episode_ids & val_episode_ids) != 0 # If true, then there is leakeage.


def split_by_episode(dataset: ReachDataset, val_frac: float = 0.1, seed: int = 42):
    '''
    - Get the sample indeces of each episode index.
    - Split sample indeces in train and validation indeces by constraining episode indeces to not overlap in these train and val sets.
    - Use sample indeces for train and val datasets and split the data with Subset().
    '''

    # get the sample id linked with each episode index.
    episode_to_indices: dict[int, list[int]] = {}
    for i, sample in enumerate(dataset.samples):
        episode_to_indices.setdefault(sample["episode_idx"], []).append(i) # key: episode idx; value: list of sample's indeces.

    print(episode_to_indices)
    episode_ids = sorted(episode_to_indices.keys())

    rng = np.random.default_rng(seed)
    rng.shuffle(episode_ids)

    # select different episodes for train dataset and validation dataset
    n_val_ep = max(1, int(val_frac * len(episode_ids)))
    print(episode_ids[n_val_ep:])
    train_episodes = set(episode_ids[n_val_ep:])
    val_episodes = set(episode_ids[:n_val_ep])

    # sample indices, i.e., indices of dataset[idx].
    train_indices = [i for ep in train_episodes for i in episode_to_indices[ep]]
    val_indices   = [i for ep in val_episodes   for i in episode_to_indices[ep]]

    return Subset(dataset, train_indices), Subset(dataset, val_indices)

In [ ]:
hdf5_path = Path.cwd() / "data" / "demos.hdf5"
dataset = ReachDataset(hdf5_path)

train_set, val_set = split_by_episode(dataset, val_frac=0.1, seed=42) # with seed=20 episode 0 is in val_indices

if _check_intersection_episodes(train_set, val_set):
    raise ValueError("Intersection detected between episodes indices in train_set and val_set")

train_loader = DataLoader(train_set, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_set, batch_size=16, shuffle=False, collate_fn=collate_fn)


{0: [0, 1, 2, 3, 4, 5, 6, 7], 1: [8, 9, 10, 11, 12, 13, 14, 15], 2: [16, 17, 18, 19, 20]}
[1, 0]
train episodes: {0, 1} 		 val episode: {2}


### C5. Baseline: direct regression policy

#### model: predict trajectory (ee_pos), based on inputs: rgb, proprio and depth.
- Inputs: rgb, proprio and depth.
- Outputs: trajectory (ee_pos)